# Notebook 03 — Pre-entrenamiento MLM sobre CDS de conotoxinas

**Bootcamp Bio-LLMs · Módulo 1 · Sesión 3 de 3**
Proyecto posdoctoral CICESE — Modelos de lenguaje para venómica integrativa de *Conus*.

---

## Objetivo final del Módulo 1

> *"Implementar desde cero, en PyTorch puro, un bloque encoder de 2 capas y entrenarlo en un toy task de predicción de codones enmascarados sobre ~10 MB de CDS de Conus. Verificar con perplejidad."*

En este notebook:

1. Generamos un **corpus sintético de CDS de precursores conotoxínicos** (~10 MB) con estructura biológicamente plausible: ATG inicial, péptido señal, propéptido, péptido maduro rico en cisteínas, codón de paro.
   * Opcional: instrucciones para reemplazar por datos reales de NCBI BioProject **PRJNA437715** / **PRJNA526781**.
2. Construimos un **tokenizador de codones** (3-mers no superpuestos sobre nucleótidos) → vocabulario 64 + 5 especiales.
3. Implementamos el **colator MLM** estilo BERT: enmascaramos 15 % de los codones (80 % MASK / 10 % aleatorio / 10 % sin cambio).
4. Conectamos el encoder del **Notebook 02** a una **cabeza MLM** (peso atado con el embedding).
5. Entrenamos en GPU (~5–15 min en A100) y reportamos **pérdida MLM, accuracy y perplejidad** sobre un set de validación holdout.
6. Inspeccionamos qué aprendió: matriz de confusión codón-a-codón, atención sobre cisteínas, predicciones top-k.

## Pre-requisitos

* Notebooks 01 y 02 ejecutados (reutilizamos `TinyEncoder`).
* GPU recomendada (A100 ideal pero T4/L4/CPU funcionan a menor escala).

## 0. Imports y configuración

In [ ]:
import math
import random
import string
from pathlib import Path
from collections import Counter
from typing import List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} | {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

OUTPUT_DIR = Path("./outputs_modulo1")
OUTPUT_DIR.mkdir(exist_ok=True)
DATA_PATH = OUTPUT_DIR / "conus_synthetic_cds.txt"

## 1. Generación del corpus sintético

### ¿Por qué sintético?

Para que este notebook sea **autocontenido y reproducible** generamos un corpus que:
* Sigue las restricciones biológicas reales de un precursor conotoxínico (estructura de tres dominios documentada en Jin et al. 2019).
* Tiene la riqueza estadística suficiente para que el MLM aprenda algo no-trivial.
* Pesa ~10 MB, suficiente para ver perplejidad bajar consistentemente en un A100 sin saturar la VRAM.

### Estructura del precursor (Jin et al. 2019)

```
   ATG     [péptido señal ~20 aa]   [propéptido ~25 aa]   [péptido maduro 10-40 aa, rico en C]   STOP
```

* **Péptido señal**: enriquecido en residuos hidrofóbicos (L, V, I, A, F).
* **Propéptido**: con sitios de procesamiento KR/KK característicos.
* **Péptido maduro**: marca registrada de la superfamilia — patrón cisteínico CC-C-C (framework I) o similar.

### Para datos reales

Reemplaza la celda generadora por:

```bash
# Datasets reales del proyecto
datasets download genome accession PRJNA437715 --include cds
datasets download genome accession PRJNA526781 --include cds
```

o, en Python con `Biopython`:

```python
from Bio import Entrez, SeqIO
Entrez.email = "tu_correo@cicese.mx"
handle = Entrez.efetch(db="nucleotide", id=lista_de_acc, rettype="fasta_cds_na", retmode="text")
records = list(SeqIO.parse(handle, "fasta"))
```

In [ ]:
# Codones por tipo de residuo (subconjunto realista)
HYDROPHOBIC_CODONS = {  # péptido señal
    "L": ["CTG", "CTT", "TTA"], "V": ["GTG", "GTT"], "I": ["ATC", "ATT"],
    "A": ["GCT", "GCC"], "F": ["TTT", "TTC"], "M": ["ATG"], "G": ["GGT"],
}
CHARGED_CODONS = {  # propéptido
    "K": ["AAA", "AAG"], "R": ["AGA", "CGT"], "D": ["GAT", "GAC"],
    "E": ["GAA", "GAG"], "H": ["CAT", "CAC"], "Q": ["CAA"], "N": ["AAT"],
}
MATURE_CODONS = {  # péptido maduro: rico en cisteínas
    "C": ["TGT", "TGC"],   # cisteína: enriquecida
    "P": ["CCA", "CCT", "CCG"],
    "G": ["GGA", "GGC"],
    "S": ["TCA", "AGC", "TCC"],
    "T": ["ACA", "ACC"],
    "Y": ["TAT", "TAC"],
    "R": ["CGT", "AGA"],
    "H": ["CAT"],
    "N": ["AAT", "AAC"],
    "K": ["AAA"],
    "D": ["GAT"],
    "E": ["GAG"],
}
STOP_CODONS = ["TAA", "TAG", "TGA"]


def sample_codon(codon_dict, weights=None):
    aa = random.choices(list(codon_dict.keys()), weights=weights)[0]
    return random.choice(codon_dict[aa])


def synthesize_signal_peptide(length=20):
    """Péptido señal: 20 residuos hidrofóbicos."""
    return "".join(sample_codon(HYDROPHOBIC_CODONS) for _ in range(length))


def synthesize_propeptide(length=25):
    """Propéptido con sitio de procesamiento KR/KK al final."""
    cds = "".join(sample_codon(CHARGED_CODONS) for _ in range(length - 2))
    cleavage = random.choice(["AAACGT", "AAAAAA", "CGTAAA"])  # KR, KK, RK
    return cds + cleavage


def synthesize_mature_peptide(framework="I"):
    """
    Péptido maduro con framework cisteínico realista.
    Framework I (α-conotoxinas): CC-X(3-5)-C-X(3-5)-C, 12-15 aa
    Framework III (μ/ω-conotoxinas): CC-X(3-5)-C-X(3-5)-CC, 16-22 aa
    """
    cys = "TGT"  # codón de cisteína
    if framework == "I":
        parts = [
            cys, cys,  # CC
            "".join(sample_codon(MATURE_CODONS) for _ in range(random.randint(3, 5))),
            cys,
            "".join(sample_codon(MATURE_CODONS) for _ in range(random.randint(3, 5))),
            cys,
        ]
    else:  # framework III
        parts = [
            cys, cys,
            "".join(sample_codon(MATURE_CODONS) for _ in range(random.randint(3, 5))),
            cys,
            "".join(sample_codon(MATURE_CODONS) for _ in range(random.randint(3, 5))),
            cys, cys,
        ]
    return "".join(parts)


def synthesize_precursor():
    """Un CDS completo: ATG + signal + propeptide + mature + STOP."""
    framework = random.choice(["I", "III"])
    cds = (
        "ATG"
        + synthesize_signal_peptide(length=random.randint(18, 22))
        + synthesize_propeptide(length=random.randint(22, 28))
        + synthesize_mature_peptide(framework=framework)
        + random.choice(STOP_CODONS)
    )
    return cds


# Generar corpus (~10 MB ≈ 50 000 precursores promedio 200 nt)
print("Generando corpus sintético…")
TARGET_BYTES = 10 * 1024 * 1024  # 10 MB
sequences = []
total_bytes = 0
while total_bytes < TARGET_BYTES:
    seq = synthesize_precursor()
    sequences.append(seq)
    total_bytes += len(seq) + 1

with open(DATA_PATH, "w") as f:
    for seq in sequences:
        f.write(seq + "\n")

n_seqs = len(sequences)
total_mb = DATA_PATH.stat().st_size / 1e6
print(f"Generadas {n_seqs:,} secuencias  |  {total_mb:.2f} MB")
print(f"Longitud media: {np.mean([len(s) for s in sequences]):.1f} nt")
print(f"Longitud min/max: {min(len(s) for s in sequences)} / {max(len(s) for s in sequences)}")
print(f"\nEjemplo:\n  {sequences[0]}")

## 2. Tokenizador de codones (3-mers no superpuestos)

* **Vocabulario**: 64 codones + 5 tokens especiales.
* Tokens especiales: `[PAD]=0, [MASK]=1, [CLS]=2, [SEP]=3, [UNK]=4`.
* Codones (64) ocupan los índices 5 → 68.

Esta es la versión más sencilla; en el Módulo 2 compararás con BPE y k-mers superpuestos.

In [ ]:
class CodonTokenizer:
    """Tokenizador no-superpuesto a 3-mers (codones), estilo MLM."""

    SPECIAL_TOKENS = ["[PAD]", "[MASK]", "[CLS]", "[SEP]", "[UNK]"]

    def __init__(self):
        # Construir vocabulario completo de codones
        bases = "ACGT"
        codons = [a + b + c for a in bases for b in bases for c in bases]  # 64
        self.vocab = self.SPECIAL_TOKENS + codons
        self.token_to_id = {tok: i for i, tok in enumerate(self.vocab)}
        self.id_to_token = {i: tok for tok, i in self.token_to_id.items()}

        # IDs útiles
        self.pad_id  = self.token_to_id["[PAD]"]
        self.mask_id = self.token_to_id["[MASK]"]
        self.cls_id  = self.token_to_id["[CLS]"]
        self.sep_id  = self.token_to_id["[SEP]"]
        self.unk_id  = self.token_to_id["[UNK]"]

        # IDs de codones (sólo los 64 reales, sin especiales) — útil para reemplazos aleatorios
        self.codon_ids = list(range(len(self.SPECIAL_TOKENS), len(self.vocab)))

    @property
    def vocab_size(self):
        return len(self.vocab)

    def encode(self, sequence: str, max_len: int = 128, add_special: bool = True) -> List[int]:
        """Convierte una secuencia de nucleótidos en una lista de IDs."""
        # Saltar nucleótidos finales que no formen codón completo
        n_codons = len(sequence) // 3
        codons = [sequence[3*i : 3*i+3].upper() for i in range(n_codons)]

        ids = [self.token_to_id.get(c, self.unk_id) for c in codons]

        if add_special:
            ids = [self.cls_id] + ids + [self.sep_id]

        # Truncar
        ids = ids[:max_len]
        # Pad
        ids = ids + [self.pad_id] * (max_len - len(ids))

        return ids

    def decode(self, ids: List[int]) -> str:
        return " ".join(self.id_to_token[i] for i in ids)


# Test
tokenizer = CodonTokenizer()
print(f"Vocab size : {tokenizer.vocab_size}")
print(f"Especiales : {tokenizer.SPECIAL_TOKENS}  → IDs {[tokenizer.token_to_id[t] for t in tokenizer.SPECIAL_TOKENS]}")
print()
example = sequences[0]
ids = tokenizer.encode(example, max_len=40)
print(f"Secuencia : {example[:60]}...")
print(f"Tokens    : {ids[:20]}...")
print(f"Decoded   : {tokenizer.decode(ids[:15])}")

## 3. Dataset y collator MLM

El **collator** implementa la estrategia BERT canónica:
* Selecciona aleatoriamente el **15 %** de los tokens (excluyendo especiales).
* De ese 15 %:
  * 80 % → `[MASK]`
  * 10 % → token aleatorio del vocabulario
  * 10 % → se deja igual

Esto fuerza al modelo a producir buenas representaciones para *todos* los tokens, no sólo los marcados con `[MASK]`.

In [ ]:
class ConotoxinDataset(Dataset):
    def __init__(self, sequences: List[str], tokenizer: CodonTokenizer, max_len: int = 128):
        self.sequences = sequences
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        ids = self.tokenizer.encode(self.sequences[idx], max_len=self.max_len)
        return torch.tensor(ids, dtype=torch.long)


def mlm_collator(batch: List[torch.Tensor], tokenizer: CodonTokenizer,
                 mask_prob: float = 0.15) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Recibe lista de tensores de tokens y devuelve:
    * inputs : tokens con enmascaramiento aplicado (B, N)
    * labels : tokens originales en las posiciones enmascaradas, -100 en el resto (B, N)
              (la convención -100 hace que CrossEntropyLoss los ignore automáticamente)
    """
    inputs = torch.stack(batch)
    labels = inputs.clone()

    # Máscara de "candidatos a enmascarar": NO especiales NI padding
    special_ids_tensor = torch.tensor(
        [tokenizer.pad_id, tokenizer.cls_id, tokenizer.sep_id, tokenizer.mask_id]
    )
    candidate = ~torch.isin(inputs, special_ids_tensor)

    # Decisión Bernoulli(0.15) sobre los candidatos
    probs = torch.full(inputs.shape, mask_prob)
    probs.masked_fill_(~candidate, 0.0)
    selected = torch.bernoulli(probs).bool()

    # Etiquetas: -100 donde NO se enmascaró
    labels[~selected] = -100

    # Sub-muestreo dentro del 15 %:
    # 80 % -> [MASK]
    replace_with_mask = torch.bernoulli(torch.full(inputs.shape, 0.8)).bool() & selected
    inputs[replace_with_mask] = tokenizer.mask_id

    # 10 % -> token aleatorio
    replace_with_random = (
        torch.bernoulli(torch.full(inputs.shape, 0.5)).bool()  # 50% del 20% restante
        & selected
        & ~replace_with_mask
    )
    random_tokens = torch.tensor(
        np.random.choice(tokenizer.codon_ids, size=inputs.shape)
    )
    inputs[replace_with_random] = random_tokens[replace_with_random]

    # 10 % restante -> sin cambio (ya está así)
    return inputs, labels


# --- Test del collator ---
ds_demo = ConotoxinDataset(sequences[:8], tokenizer, max_len=64)
batch = [ds_demo[i] for i in range(4)]
inp, lbl = mlm_collator(batch, tokenizer)

n_total = (lbl != -100).sum().item()
n_mask = (inp == tokenizer.mask_id).sum().item()
print(f"Tokens etiquetados (enmascarados): {n_total}")
print(f"De ellos con [MASK]              : {n_mask}  ({n_mask/n_total*100:.1f} %, esperado ~80 %)")

## 4. Modelo: encoder + cabeza MLM

Importamos el `TinyEncoder` del Notebook 02 y le añadimos una cabeza de predicción de codones. Atamos el peso de la cabeza al de la matriz de embedding (*weight tying*, técnica estándar que reduce parámetros y mejora generalización).

In [ ]:
# --- Reimport mínimo del Notebook 02 (en producción esto vendría de un módulo) ---
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask, float("-inf"))
    attn = F.softmax(scores, dim=-1)
    return torch.matmul(attn, V), attn


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.0):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model, self.num_heads = d_model, num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        B, N, _ = x.shape
        Q = self.W_q(x).view(B, N, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, N, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, N, self.num_heads, self.d_k).transpose(1, 2)
        out, attn = scaled_dot_product_attention(Q, K, V, mask=mask)
        out = out.transpose(1, 2).contiguous().view(B, N, self.d_model)
        return self.dropout(self.W_o(out)), attn


class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.0):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.linear2(self.dropout(F.gelu(self.linear1(x))))


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, num_heads, dropout=dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout=dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, mask=None):
        # Pre-norm
        attn_out, attn = self.mha(self.norm1(x), mask=mask)
        x = x + self.dropout(attn_out)
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x, attn


# --- El modelo MLM completo ---
class TinyMLM(nn.Module):
    def __init__(self, vocab_size, d_model=128, num_heads=4, num_layers=2,
                 d_ff=512, max_len=256, dropout=0.1, pad_id=0, tie_weights=True):
        super().__init__()
        self.pad_id = pad_id
        self.d_model = d_model

        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_encoding = SinusoidalPositionalEncoding(d_model, max_len=max_len)
        self.embed_dropout = nn.Dropout(dropout)

        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout=dropout)
            for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)

        # Cabeza MLM
        self.mlm_head = nn.Linear(d_model, vocab_size, bias=True)
        if tie_weights:
            self.mlm_head.weight = self.token_embedding.weight  # weight tying

    def forward(self, token_ids):
        mask = (token_ids == self.pad_id).unsqueeze(1).unsqueeze(2)
        x = self.token_embedding(token_ids) * math.sqrt(self.d_model)
        x = self.embed_dropout(self.pos_encoding(x))
        for layer in self.layers:
            x, _ = layer(x, mask=mask)
        x = self.final_norm(x)
        return self.mlm_head(x)  # (B, N, vocab_size)


# Construir el modelo
config = dict(
    vocab_size=tokenizer.vocab_size,
    d_model=128,
    num_heads=4,
    num_layers=2,
    d_ff=512,
    max_len=128,
    dropout=0.1,
    pad_id=tokenizer.pad_id,
)
model = TinyMLM(**config).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"TinyMLM construido:")
for k, v in config.items():
    print(f"  {k:<12} = {v}")
print(f"\nParámetros entrenables: {n_params:,}  (~{n_params/1e6:.3f} M)")

## 5. Entrenamiento

Hiperparámetros pedagógicos (no es el óptimo, es lo necesario para ver la curva bajar en pocos minutos):

* Optimizer: **AdamW** (lr 5e-4, β=(0.9, 0.999), weight_decay=0.01).
* Scheduler: warmup lineal 500 pasos + decaimiento coseno.
* Mixed precision: `torch.cuda.amp.autocast` si hay GPU.
* Gradient clipping: 1.0.
* Eval cada 500 steps.

In [ ]:
# Split train / val
random.shuffle(sequences)
n_val = max(500, int(0.05 * len(sequences)))
train_seqs = sequences[:-n_val]
val_seqs   = sequences[-n_val:]
print(f"Train: {len(train_seqs):,}  |  Val: {len(val_seqs):,}")

train_ds = ConotoxinDataset(train_seqs, tokenizer, max_len=128)
val_ds   = ConotoxinDataset(val_seqs,   tokenizer, max_len=128)

def collate_fn(batch):
    return mlm_collator(batch, tokenizer)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,
                          collate_fn=collate_fn, num_workers=0, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=128, shuffle=False,
                          collate_fn=collate_fn, num_workers=0)

print(f"Batches por época (train): {len(train_loader)}")

In [ ]:
def cosine_warmup_lr(step, warmup_steps, total_steps, base_lr):
    """LR scheduler: warmup lineal seguido de decay cosenoidal."""
    if step < warmup_steps:
        return base_lr * step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * base_lr * (1 + math.cos(math.pi * progress))


@torch.no_grad()
def evaluate(model, loader, device):
    """Devuelve (loss media, accuracy en posiciones enmascaradas, perplejidad)."""
    model.eval()
    total_loss, total_correct, total_tokens = 0.0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        logits = model(inputs)                       # (B, N, V)
        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            labels.view(-1),
            ignore_index=-100,
            reduction="sum",
        )
        valid = (labels != -100)
        n = valid.sum().item()
        if n == 0:
            continue
        total_loss += loss.item()
        preds = logits.argmax(dim=-1)
        total_correct += ((preds == labels) & valid).sum().item()
        total_tokens += n
    mean_loss = total_loss / max(1, total_tokens)
    acc = total_correct / max(1, total_tokens)
    ppl = math.exp(mean_loss)
    return mean_loss, acc, ppl


# Configuración de entrenamiento
EPOCHS = 3
BASE_LR = 5e-4
WARMUP_STEPS = 500
TOTAL_STEPS = EPOCHS * len(train_loader)
EVAL_EVERY = 500
LOG_EVERY = 100

optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR,
                              betas=(0.9, 0.999), weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

history = {"step": [], "train_loss": [], "val_loss": [], "val_acc": [], "val_ppl": []}

print(f"Pasos totales: {TOTAL_STEPS}  |  Warmup: {WARMUP_STEPS}\n")

# Evaluación inicial (baseline: 1/vocab_size accuracy esperado)
val_loss0, val_acc0, val_ppl0 = evaluate(model, val_loader, device)
print(f"[step 0]    val_loss={val_loss0:.4f}  val_acc={val_acc0:.4f}  val_ppl={val_ppl0:.2f}")
print(f"            (random baseline acc ≈ 1/{tokenizer.vocab_size} = {1/tokenizer.vocab_size:.4f})\n")

global_step = 0
running_loss = 0.0
running_n = 0

for epoch in range(EPOCHS):
    model.train()
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        # LR schedule
        lr = cosine_warmup_lr(global_step, WARMUP_STEPS, TOTAL_STEPS, BASE_LR)
        for pg in optimizer.param_groups:
            pg["lr"] = lr

        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            logits = model(inputs)
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                labels.view(-1),
                ignore_index=-100,
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * (labels != -100).sum().item()
        running_n += (labels != -100).sum().item()
        global_step += 1

        if global_step % LOG_EVERY == 0:
            train_loss = running_loss / max(1, running_n)
            print(f"[step {global_step:>5}]  lr={lr:.2e}  train_loss={train_loss:.4f}  train_ppl={math.exp(train_loss):.2f}")
            history["step"].append(global_step)
            history["train_loss"].append(train_loss)
            running_loss, running_n = 0.0, 0

        if global_step % EVAL_EVERY == 0 or global_step == TOTAL_STEPS:
            val_loss, val_acc, val_ppl = evaluate(model, val_loader, device)
            print(f"           ↳ VAL  loss={val_loss:.4f}  acc={val_acc:.4f}  ppl={val_ppl:.2f}")
            history["val_loss"].append((global_step, val_loss))
            history["val_acc"].append((global_step, val_acc))
            history["val_ppl"].append((global_step, val_ppl))
            model.train()

print("\n✓ Entrenamiento terminado.")

## 6. Curvas de aprendizaje y diagnóstico de perplejidad

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Train loss
axes[0].plot(history["step"], history["train_loss"], label="train", alpha=0.7)
if history["val_loss"]:
    vs, vl = zip(*history["val_loss"])
    axes[0].plot(vs, vl, "o-", label="val", color="red")
axes[0].set_xlabel("step"); axes[0].set_ylabel("CE loss")
axes[0].set_title("Pérdida")
axes[0].legend(); axes[0].grid(alpha=0.3)

# Perplexity
if history["val_ppl"]:
    vs, vp = zip(*history["val_ppl"])
    axes[1].plot(vs, vp, "o-", color="purple")
    axes[1].axhline(tokenizer.vocab_size, ls="--", color="gray",
                    label=f"baseline (uniforme) = {tokenizer.vocab_size}")
    axes[1].set_xlabel("step"); axes[1].set_ylabel("Perplejidad")
    axes[1].set_title("Perplejidad de validación")
    axes[1].legend(); axes[1].grid(alpha=0.3)
    axes[1].set_yscale("log")

# Accuracy
if history["val_acc"]:
    vs, va = zip(*history["val_acc"])
    axes[2].plot(vs, va, "o-", color="green")
    axes[2].axhline(1/tokenizer.vocab_size, ls="--", color="gray", label="random")
    axes[2].set_xlabel("step"); axes[2].set_ylabel("Accuracy")
    axes[2].set_title("Accuracy en posiciones enmascaradas")
    axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. ¿Qué aprendió el modelo?

Tres diagnósticos clásicos:

1. **Distribución de la pérdida por tipo de codón**: ¿predice mejor codones en regiones del péptido maduro (con motivos como CC-X-C-X-C) que en el péptido señal aleatorio?
2. **Top-k accuracy**: además de top-1, mira top-3 y top-5.
3. **Predicciones ejemplo**: enmascara una cisteína dentro de un patrón y observa la distribución de salida — ¿le da alta probabilidad a TGT/TGC?

In [ ]:
@torch.no_grad()
def top_k_accuracy(model, loader, device, ks=(1, 3, 5)):
    model.eval()
    counts = {k: 0 for k in ks}
    total = 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        logits = model(inputs)
        valid = labels != -100
        if valid.sum() == 0: continue
        n = valid.sum().item()
        total += n
        for k in ks:
            topk = logits.topk(k, dim=-1).indices  # (B, N, k)
            correct = (topk == labels.unsqueeze(-1)).any(dim=-1)
            counts[k] += (correct & valid).sum().item()
    return {k: counts[k] / max(1, total) for k in ks}


accs = top_k_accuracy(model, val_loader, device)
print("Top-k accuracy en validación:")
for k, v in accs.items():
    print(f"  top-{k}: {v:.4f}")

In [ ]:
# Predicción puntual: enmascarar una cisteína en un motivo CC-...
example_cds = "ATG" + "TGT" + "TGT" + "GCA" + "TGT" + "TAA"
# CDS:           Met   Cys   Cys   Ala   Cys   STOP
example_ids = tokenizer.encode(example_cds, max_len=20)
masked = example_ids.copy()

# Encontrar la última cisteína (TGT) y enmascararla
target_idx = None
for i, tid in enumerate(example_ids):
    if tokenizer.id_to_token[tid] == "TGT":
        target_idx = i
        last_cys_id = tid

# Enmascaramos la segunda cisteína (índice 2 de tokens reales, considerando CLS al inicio)
target_idx = 4  # posición de la última TGT antes del STOP
gold_token = tokenizer.id_to_token[masked[target_idx]]
masked[target_idx] = tokenizer.mask_id

inputs = torch.tensor([masked], device=device)
model.eval()
with torch.no_grad():
    logits = model(inputs)[0, target_idx]   # (vocab_size,)
    probs = F.softmax(logits, dim=-1)

# Top 10 predicciones
top_probs, top_ids = probs.topk(10)
print(f"Contexto: {tokenizer.decode(example_ids[:8])}")
print(f"Codón enmascarado en posición {target_idx} (verdadero = {gold_token})\n")
print(f"{'Rank':<6}{'Codón':<10}{'Prob':<10}{'¿Cisteína?'}")
print("-" * 40)
for rank, (p, i) in enumerate(zip(top_probs.cpu().numpy(), top_ids.cpu().numpy())):
    tok = tokenizer.id_to_token[int(i)]
    is_cys = "✓" if tok in ("TGT", "TGC") else ""
    print(f"{rank+1:<6}{tok:<10}{p:.4f}    {is_cys}")

## 8. Guardar checkpoint y métricas finales

In [ ]:
# Métricas finales
final_loss, final_acc, final_ppl = evaluate(model, val_loader, device)
print(f"=== Métricas finales (validación) ===")
print(f"  Loss        : {final_loss:.4f}")
print(f"  Accuracy    : {final_acc:.4f}")
print(f"  Perplejidad : {final_ppl:.2f}")
print(f"  Baseline    : {tokenizer.vocab_size:.0f} (uniforme)\n")

# Guardar
ckpt_path = OUTPUT_DIR / "tinymlm_modulo1.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "config": config,
    "tokenizer_vocab": tokenizer.vocab,
    "final_metrics": {"loss": final_loss, "acc": final_acc, "ppl": final_ppl},
    "history": history,
}, ckpt_path)
print(f"✓ Checkpoint guardado en {ckpt_path}  ({ckpt_path.stat().st_size/1e6:.2f} MB)")

## 9. Conclusión del Módulo 1

✓ Has implementado **desde cero**, sin `nn.Transformer`:
* Scaled dot-product attention y multi-head attention (Notebook 01).
* Codificación posicional sinusoidal, FFN, LayerNorm, residuales, bloque encoder completo (Notebook 02).
* Tokenizador de codones, collator MLM estilo BERT, modelo TinyMLM con weight tying, loop de entrenamiento con AMP, evaluación con perplejidad (Notebook 03).

### ¿La perplejidad bajó? Interpretación

* **Baseline uniforme**: ppl = `vocab_size` = 69 (un modelo que adivina al azar).
* **Buen entrenamiento toy**: ppl entre 5 y 25 dependiendo del tiempo de entrenamiento.
* **DNABERT real (humano completo)**: ppl ≈ 2-4 en validación, pero entrenado durante 25 días en 8× 2080Ti.

Si tu ppl está cerca del baseline (~60), revisa: ¿se aplicó el MLM correctamente? ¿La pérdida bajó en train pero no en val (overfitting)?

### Conexión con los Módulos 2-7

| Lo aprendido aquí | Sirve directamente en |
|---|---|
| Pre-entrenamiento MLM | Módulo 3 (escalar a A100 con FlashAttention y BF16) |
| Tokenizador de codones | Módulo 2 (comparar contra BPE y k-meros de DNABERT/NT) |
| Cálculo de perplejidad | Módulo 5 (métricas intrínsecas) |
| Encoder pre-norm | Módulo 4 (donde adaptarás un encoder pre-entrenado a tareas downstream con LoRA) |

### Lectura para el Módulo 2

Devlin et al. (2019) **BERT** y Ji et al. (2021) **DNABERT** — replica los hiperparámetros de DNABERT-6 sobre este corpus y observa la diferencia.

In [ ]:
print("=== Módulo 1 completado ===")
print(f"Notebook 01: Atención desde cero")
print(f"Notebook 02: Bloque encoder completo")
print(f"Notebook 03: Pre-entrenamiento MLM (este notebook)\n")
print(f"Próximo: Módulo 2 — Tokenización biológica (k-mer vs BPE vs H-Net)")